# Step 1 — 篩選文章
從 `stock_text` 撈出與目標公司相關的主文，輸出 `articles_raw.csv`

Big Data & Business Analytics, National Taiwan University

In [1]:
# ── 可調整參數區 ──────────────────────────────────────
# --- 設定區域 ---
COMPANY_ID = '1519'
COMPANY_NAME = '華城'

# 擴充關鍵字 (包含詞)
INCLUDE_KEYWORDS = [
    '華城', '華城電機', '華城電能', 'EVALUE', 
    'Fortune Electric', '重電三雄', '重電四君子'
]

# 排除關鍵字 (避免京華城弊案噪音)
EXCLUDE_KEYWORDS = [
    '京華城', '柯文哲', '橘子', '許芷瑜', 
    '沈慶京', '應曉薇', '容積率', '弊案', '羈押'
]
# ──────────────────────────────────────────────────────

import os
from dotenv import load_dotenv
load_dotenv()  # 讀取同目錄下的 .env 檔案

DB_CONFIG = {
    'host':     os.getenv('DB_HOST',    'localhost'),
    'user':     os.getenv('DB_USER',    'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME',    'bda2026'),
    'charset':  os.getenv('DB_CHARSET', 'utf8mb4')
}

In [2]:
import pymysql
import csv

In [3]:
# 連線資料庫，取得 stock_prices 的日期範圍
conn = pymysql.connect(**DB_CONFIG)
cursor = conn.cursor()

cursor.execute("""
    SELECT MIN(trade_date), MAX(trade_date)
    FROM stock_prices
    WHERE company_id = %s
""", (COMPANY_ID,))
min_date, max_date = cursor.fetchone()
print(f'股價資料範圍：{min_date} ～ {max_date}')



股價資料範圍：2023-03-01 ～ 2025-02-27


In [4]:
# [METHOD] 目前方法：動態生成 LIKE 多條件過濾 (包含擴充字與排除雜訊)

# 1. 將 COMPANY_ID (1519) 加入包含名單一起比對
search_keywords = INCLUDE_KEYWORDS + [COMPANY_ID]

# 2. 動態建構包含條件：(title LIKE %s OR content LIKE %s OR ...)
include_clause = " OR ".join(["(title LIKE %s OR content LIKE %s)"] * len(search_keywords))
include_params = []
for kw in search_keywords:
    include_params.extend([f'%{kw}%', f'%{kw}%'])

# #只查詢標題
# include_clause = " OR ".join(["title LIKE %s"] * len(search_keywords))
# include_params = []
# for kw in search_keywords:
#     include_params.append(f'%{kw}%') # 只 append 一次

# 3. 動態建構排除條件：(title NOT LIKE %s AND content NOT LIKE %s AND ...)
if EXCLUDE_KEYWORDS:
    exclude_clause = " AND " + " AND ".join(["(title NOT LIKE %s AND content NOT LIKE %s)"] * len(EXCLUDE_KEYWORDS))
    exclude_params = []
    for kw in EXCLUDE_KEYWORDS:
        exclude_params.extend([f'%{kw}%', f'%{kw}%'])
else:
    exclude_clause = ""
    exclude_params = []

# 4. 組合最終 SQL
sql_query = f"""
    SELECT no, post_time, title, content, s_name
    FROM stock_text
    WHERE ({include_clause})
      {exclude_clause}
      AND (content_type = 'main' OR content_type IS NULL)
      AND DATE(post_time) >= %s
      AND DATE(post_time) <= %s
    ORDER BY post_time
"""

# 5. 參數組合順序：包含詞 -> 排除詞 -> 時間範圍
final_params = include_params + exclude_params + [min_date, max_date]

print("正在進行深度篩選與去噪，請稍候...")
cursor.execute(sql_query, final_params)

rows = cursor.fetchall()
print(f'精確篩選後，找到 {len(rows)} 篇相關文章')

# 關閉連線
cursor.close()
conn.close()

正在進行深度篩選與去噪，請稍候...
精確篩選後，找到 4105 篇相關文章


In [6]:
import re

boilerplate_titles = ['買賣超排行', '投信買賣', '外資買賣', '主力買賣', '法人買賣']

def is_boilerplate(title, content):
    text = (title or '') + (content or '')
    # 標題是排行類
    for pattern in boilerplate_titles:
        if pattern in (title or ''):
            return True
    # 數字比例超過 30%
    if len(text) > 0 and sum(c.isdigit() for c in text) / len(text) > 0.3:
        return True
    # 文章太短
    if len(text) < 50:
        return True
    return False

# 套用篩選
# rows 欄位順序：no, post_time, title, content, s_name
rows_kept    = [r for r in rows if not is_boilerplate(r[2], r[3])]
rows_removed = [r for r in rows if     is_boilerplate(r[2], r[3])]

print(f'篩選前總筆數：{len(rows)}')
print(f'篩選掉筆數：  {len(rows_removed)}')
print(f'保留筆數：    {len(rows_kept)}')

篩選前總筆數：4105
篩選掉筆數：  800
保留筆數：    3305


In [7]:
HEADER = ['no', 'post_time', 'title', 'content', 's_name']

# 保留的文章（供後續 step2 使用）
with open('articles_raw.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(HEADER)
    writer.writerows(rows_kept)
print(f'已儲存 articles_raw.csv（保留）：{len(rows_kept)} 筆')

# 被篩選掉的文章（供人工檢查）
with open('articles_filtered_out.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(HEADER)
    writer.writerows(rows_removed)
print(f'已儲存 articles_filtered_out.csv（過濾）：{len(rows_removed)} 筆')

已儲存 articles_raw.csv（保留）：3305 筆
已儲存 articles_filtered_out.csv（過濾）：800 筆


In [9]:
from collections import Counter

def print_source_dist(row_list, label):
    print(f'\n===== {label}（共 {len(row_list)} 筆）=====')
    src_count = Counter(r[4] for r in row_list)
    for src, cnt in src_count.most_common():
        print(f'  {src}: {cnt} 篇')

print_source_dist(rows_kept,    '保留文章 - 各平台分布')
print_source_dist(rows_removed, '過濾文章 - 各平台分布')


===== 保留文章 - 各平台分布（共 3305 筆）=====
  Yahoo股市: 2363 篇
  Yahoo新聞: 347 篇
  Ptt: 335 篇
  Dcard: 255 篇
  Mobile01: 5 篇

===== 過濾文章 - 各平台分布（共 800 筆）=====
  Ptt: 523 篇
  Yahoo股市: 222 篇
  Dcard: 29 篇
  Yahoo新聞: 26 篇


In [35]:
import pandas as pd

df = pd.read_csv('articles_raw.csv')

# 每個來源各抽 3 篇，看標題和內文前 100 字
for source, group in df.groupby('s_name'):
    print(f'\n{"="*50}')
    print(f'來源：{source}')
    print(f'{"="*50}')
    for _, row in group.sample(min(3, len(group)), random_state=42).iterrows():
        print(f'\n標題：{row["title"]}')
        print(f'內文：{str(row["content"])[:]}')
        print('-'*30)


來源：Dcard

標題：04/12 台股收盤指數
內文：⚙️ 大盤收盤統計 ⚙️ 

📌 加權指數
15932.97 / 19.09(0.12%)
📌 櫃買指數
216.61 / 0.9(0.42%)
📌 台股成交金額
2289.65 億


🚀 今日漲幅榜 
(代號 / 股名 / 漲幅) :
1503 / 士電 / 10.0%
1514 / 亞力 / 10.0%
1519 / 華城 / 10.0%
1529 / 樂事綠能 / 10.0%
3308 / 聯德 / 10.0%
2206 / 三陽工業 / 9.9%
2634 / 漢翔 / 9.9%


💰 交易量 TOP 5 
(代號 / 股名 / 收盤價 / 交易量(張)) :
1513 / 中興電 / 115.00 / 96336
2634 / 漢翔 / 48.15 / 96280
1609 / 大亞 / 28.70 / 82703
3481 / 群創 / 15.15 / 79079
00637L / 元大滬深300正2 / 15.09 / 75817


🔥 今日熱門概念股 TOP 5
(代號 / 名稱 / 成交價 / 漲跌 / 漲跌幅)

🎫電機 / 2.89%
1514 / 亞力 / 46.20 / ▲4.20 / +10.00%
1519 / 華城 / 96.80 / ▲8.8 / +10.00%
1503 / 士電 / 96.00 / ▲8.7 / +9.97%

🎫電器電纜 / 1.55%
1611 / 中電 / 17.10 / ▲0.95 / +5.88%
1609 / 大亞 / 28.70 / ▲1.35 / +4.94%
1618 / 合機 / 18.70 / ▲0.85 / +4.76%

🎫生技醫療 / 1.44%
6782 / 視陽 / 299.00 / ▲18.0 / +6.41%
1752 / 南光 / 62.20 / ▲2.8 / +4.71%
1707 / 葡萄王 / 184.00 / ▲6.5 / +3.66%

🎫玻璃 / 1.32%
1810 / 和成 / 19.00 / ▲0.40 / +2.15%
1802 / 台玻 / 20.45 / ▲0.30 / +1.49%
1809 / 中釉 / 14.15 / -- / 0.00%

🎫觀光 / 1.2%
2723 / 美食